# Retail Sales Database (SQLite)

In [ ]:
# Import required Python libraries
import sqlite3, random, math, os
from datetime import date, datetime, timedelta

# Set random seed to ensure reproducible random data
random.seed(2025)

# Define database name and dataset date range
DB = "retail_sales.db"
START, END = date(2024, 1, 1), date(2025, 12, 31)

# Remove existing database to create a fresh database
if os.path.exists(DB): os.remove(DB)

# Establish connection to SQLite database
con = sqlite3.connect(DB)


# Enable foreign key constraints
con.execute("PRAGMA foreign_keys = ON")

# CREATE DATABASE TABLES

con.executescript("""
CREATE TABLE stores (
    store_id       INTEGER PRIMARY KEY,
    store_name     TEXT NOT NULL UNIQUE,
    city           TEXT NOT NULL,
    nation         TEXT NOT NULL CHECK (nation IN ('England','Scotland','Wales','Northern Ireland')),  -- nominal
    postcode_area  TEXT NOT NULL,
    size_tier      TEXT NOT NULL CHECK (size_tier IN ('Small','Medium','Large','Flagship')),          -- ordinal
    floor_area_sqm REAL NOT NULL CHECK (floor_area_sqm > 0),                                          -- ratio
    opened_date    TEXT NOT NULL                                                                      -- interval
);
CREATE TABLE customers (
    customer_id       INTEGER PRIMARY KEY,
    first_name        TEXT NOT NULL,
    last_name         TEXT NOT NULL,
    gender            TEXT NOT NULL CHECK (gender IN ('Female','Male','Prefer not to say')),
    birth_year        INTEGER NOT NULL CHECK (birth_year BETWEEN 1930 AND 2007),                      -- interval
    city              TEXT NOT NULL,
    postcode_area     TEXT NOT NULL,
    email             TEXT NOT NULL UNIQUE,
    member_tier       TEXT NOT NULL CHECK (member_tier IN ('Standard','Silver','Gold','Platinum')),   -- ordinal
    signup_date       TEXT NOT NULL,
    marketing_consent INTEGER NOT NULL CHECK (marketing_consent IN (0,1))
);
CREATE TABLE products (
    product_id     INTEGER PRIMARY KEY,
    sku            TEXT NOT NULL UNIQUE,
    product_name   TEXT NOT NULL,
    category       TEXT NOT NULL,
    brand          TEXT NOT NULL,
    unit_cost_gbp  REAL NOT NULL CHECK (unit_cost_gbp > 0),                                           -- ratio
    list_price_gbp REAL NOT NULL,
    weight_kg      REAL NOT NULL CHECK (weight_kg > 0),
    CHECK (list_price_gbp >= unit_cost_gbp)
);
CREATE TABLE orders (
    order_id            INTEGER PRIMARY KEY,
    customer_id         INTEGER REFERENCES customers(customer_id),     -- NULL = guest checkout
    store_id            INTEGER NOT NULL REFERENCES stores(store_id),
    order_datetime      TEXT NOT NULL,                                 -- interval
    payment_method      TEXT NOT NULL CHECK (payment_method IN
                        ('Contactless Card','Chip & PIN Card','Cash','Mobile Wallet','Gift Card')),   -- nominal
    temperature_c       REAL,                                          -- interval (degrees Celsius)
    satisfaction_rating INTEGER CHECK (satisfaction_rating BETWEEN 1 AND 5)                           -- ordinal
);
CREATE TABLE order_items (
    order_id       INTEGER NOT NULL REFERENCES orders(order_id),
    line_no        INTEGER NOT NULL,
    product_id     INTEGER NOT NULL REFERENCES products(product_id),
    quantity       INTEGER NOT NULL CHECK (quantity > 0),              -- ratio
    unit_price_gbp REAL NOT NULL CHECK (unit_price_gbp > 0),           -- ratio (price at time of sale)
    discount_pct   REAL NOT NULL CHECK (discount_pct BETWEEN 0 AND 100),
    PRIMARY KEY (order_id, line_no)                                    -- compound key
);
CREATE TABLE inventory (
    store_id         INTEGER NOT NULL REFERENCES stores(store_id),
    product_id       INTEGER NOT NULL REFERENCES products(product_id),
    quantity_on_hand INTEGER NOT NULL CHECK (quantity_on_hand >= 0),
    reorder_level    INTEGER NOT NULL CHECK (reorder_level >= 0),
    PRIMARY KEY (store_id, product_id)                                 -- compound key
);
CREATE INDEX idx_orders_dt ON orders(order_datetime);
CREATE INDEX idx_orders_store ON orders(store_id);
CREATE INDEX idx_items_product ON order_items(product_id);
""")

In [ ]:
# stores
# Create a list of 12 supermarket stores.
# Each store contains:
# Store name, city, nation, postcode area, size tier,
# floor area (square metres) and opening date

STORES = [  # name, city, nation, postcode area, tier, floor area (sq m), opened
 ("Marlow & Finch London Oxford St", "London",     "England",          "W1",  "Flagship", 5200, "2012-03-10"),
 ("Marlow & Finch London Canary Wharf","London",   "England",          "E14", "Large",    2800, "2016-09-22"),
 ("Marlow & Finch Manchester",       "Manchester", "England",          "M1",  "Large",    3100, "2014-05-17"),
 ("Marlow & Finch Birmingham",       "Birmingham", "England",          "B2",  "Large",    3000, "2013-10-04"),
 ("Marlow & Finch Leeds",            "Leeds",      "England",          "LS1", "Medium",   1900, "2017-04-29"),
 ("Marlow & Finch Liverpool",        "Liverpool",  "England",          "L1",  "Small",     950, "2024-09-14"),
 ("Marlow & Finch Bristol",          "Bristol",    "England",          "BS1", "Medium",   1700, "2018-11-03"),
 ("Marlow & Finch Newcastle",        "Newcastle",  "England",          "NE1", "Medium",   1650, "2019-03-16"),
 ("Marlow & Finch Glasgow",          "Glasgow",    "Scotland",         "G2",  "Large",    2600, "2015-06-27"),
 ("Marlow & Finch Edinburgh",        "Edinburgh",  "Scotland",         "EH2", "Medium",   2000, "2016-02-13"),
 ("Marlow & Finch Cardiff",          "Cardiff",    "Wales",            "CF10","Medium",   1800, "2018-08-25"),
 ("Marlow & Finch Belfast",          "Belfast",    "Northern Ireland", "BT1", "Small",    1100, "2020-10-09"),
]
con.executemany("INSERT INTO stores VALUES (NULL,?,?,?,?,?,?,?)", STORES)
STORE = {r[0]: dict(city=r[1], nation=r[2], tier=r[3], opened=date.fromisoformat(r[4]))
         for r in con.execute("SELECT store_id, city, nation, size_tier, opened_date FROM stores")}
TRAFFIC = {"Flagship": 5, "Large": 3.5, "Medium": 2, "Small": 1}
POSTCODES = {"London": ["W1","E14","SE1","N1","SW9","NW3","E8"], "Manchester": ["M1","M14","M20","M16"],
             "Birmingham": ["B2","B15","B29","B13"], "Leeds": ["LS1","LS6","LS8"], "Liverpool": ["L1","L8","L17"],
             "Bristol": ["BS1","BS6","BS8"], "Newcastle": ["NE1","NE2","NE6"], "Glasgow": ["G2","G12","G41"],
             "Edinburgh": ["EH2","EH8","EH10"], "Cardiff": ["CF10","CF11","CF24"], "Belfast": ["BT1","BT7","BT9"]}

# customers
# Define lists of first names and surnames to generate
# realistic synthetic customer names.

FIRST_F = ["Olivia","Amelia","Isla","Ava","Mia","Sophie","Grace","Freya","Poppy","Ivy","Emily","Charlotte","Lucy","Hannah","Priya","Aisha","Emma","Chloe","Ella","Jessica"]
FIRST_M = ["Oliver","George","Harry","Jack","Noah","Leo","Arthur","Oscar","Charlie","Thomas","William","James","Daniel","Samuel","Muhammad","Callum","Ben","Joe","Adam","Henry"]
LAST    = ["Smith","Jones","Taylor","Brown","Williams","Wilson","Johnson","Walker","Wright","Robinson","Thompson","White","Hughes","Edwards","Green",
           "Hall","Wood","Harris","Lewis","Martin","Jackson","Clarke","Patel","Khan","Ahmed","Singh","Cooper","Ward","Baker","Turner"]

# Define regional names to add geographical variety to
# the synthetic customer dataset.
REGIONAL = {"Scotland": (["Fiona","Angus","Isla","Callum","Ewan","Morag"], ["Campbell","Stewart","MacDonald","Murray","Reid","Robertson","Fraser"]),
            "Wales": (["Rhys","Ffion","Owen","Cerys","Gareth","Megan"], ["Davies","Evans","Morgan","Price","Jenkins","Pritchard","Rees"]),
            "Northern Ireland": (["Aoife","Siobhan","Ciaran","Niamh","Conor","Sean"], ["O'Neill","Murphy","Kelly","McCann","Doherty","Quinn","Maguire"])}

# Define a function to generate a random date between
# two specified dates.
def rand_date(a, b): return a + timedelta(days=random.randint(0, (b - a).days))

# Generate 2,500 synthetic customer records.
# Customers are assigned to cities based on the store locations.
store_ids = list(STORE)
rows, used = [], set()
for _ in range(2500):
    sid = random.choices(store_ids, [TRAFFIC[STORE[s]["tier"]] for s in store_ids])[0]
    city, nation = STORE[sid]["city"], STORE[sid]["nation"]

    # Generate customer gender using predefined probabilities.
    g = random.choices(["Female","Male","Prefer not to say"], [0.49, 0.48, 0.03])[0]


    # Select a first name based on the generated gender.
    first = random.choice(FIRST_F if g == "Female" else FIRST_M if g == "Male" else FIRST_F + FIRST_M)

    # Randomly select a surname.
    last = random.choice(LAST)

     # Introduce regional names for Scotland, Wales
    # and Northern Ireland with a 40% probability.

    if nation in REGIONAL and random.random() < 0.4:                 # regional name flavour
        rf, rl = REGIONAL[nation]
        last = random.choice(rl)
        if random.random() < 0.5: first = random.choice(rf)


    # Generate a unique synthetic email address.
    email = f"{first}.{last}{random.randint(1, 9999)}@example.com".lower().replace("'", "")

    # Check for duplicate email addresses and generate
    # another email if the address already exists.
    while email in used: email = f"{first}.{last}{random.randint(1, 99999)}@example.com".lower().replace("'", "")
    used.add(email)

     # Create the customer record containing demographic,
    # location, membership and marketing information.
    rows.append((first, last, g, int(min(2007, max(1940, random.gauss(1982, 14)))), city,
                 random.choice(POSTCODES[city]), email,
                 random.choices(["Standard","Silver","Gold","Platinum"], [0.55, 0.25, 0.15, 0.05])[0],
                 rand_date(date(2018, 1, 1), END - timedelta(days=30)).isoformat(), int(random.random() < 0.6)))

# Insert all generated customer records into the customers table.
# SQLite automatically generates the customer_id values.
con.executemany("INSERT INTO customers VALUES (NULL,?,?,?,?,?,?,?,?,?,?)", rows)

# Retrieve customer information and organise it into dictionaries.
# CUST stores membership tier and signup date for each customer.
# BY_CITY groups customer IDs according to their city.
CUST, BY_CITY = {}, {}
for cid, city, tier, su in con.execute("SELECT customer_id, city, member_tier, signup_date FROM customers"):
    CUST[cid] = (tier, date.fromisoformat(su)); BY_CITY.setdefault(city, []).append(cid)

# products
# Create a product catalogue containing seven supermarket
# categories, their popularity weights and individual products.
# Each product has a typical selling price in GBP and weight in kg.

CATALOGUE = {   # category: (popularity, perishable-ish, [(item, typical price GBP, weight kg)])
 "Food Cupboard": (5, [("Baked Beans 4 x 400g",2.4,1.6),("Porridge Oats 1kg",1.8,1),("Digestive Biscuits 400g",1.5,.4),("Tea Bags 240s",3.5,.75),
                       ("Long Grain Rice 1kg",1.9,1),("Spaghetti 500g",.95,.5),("Strawberry Jam 454g",1.6,.45),("Crisps 6 Pack",1.75,.15)]),
 "Fresh & Dairy": (5, [("Semi-Skimmed Milk 2L",1.65,2.1),("Mature Cheddar 350g",3.1,.35),("Free Range Eggs 12",2.75,.8),("Salted Butter 250g",2.2,.25),
                       ("Wholemeal Bread 800g",1.3,.8),("Pork Sausages 8 Pack",2.6,.45),("Natural Yoghurt 500g",1.1,.5),("Chicken Breast 600g",4.6,.6)]),
 "Drinks": (3.5, [("Orange Juice 1L",1.8,1),("Cola 2L",1.75,2),("Instant Coffee 200g",5.2,.2),("Ginger Beer 4 Pack",2.5,1.4),
                  ("Squash 1L",1.4,1),("Still Water 6 x 1.5L",3.2,9),("Hot Chocolate 500g",3.3,.5),("Sparkling Water 6 x 500ml",2.8,3)]),
 "Household": (2.5, [("Laundry Capsules 40",7.5,1),("Washing-Up Liquid 500ml",1.45,.55),("Kitchen Roll 4 Pack",4,1),("Toilet Tissue 9 Roll",4.75,2),
                     ("Bin Bags 40 Pack",3.2,.6),("Thick Bleach 750ml",1.2,.8),("LED Light Bulb 2 Pack",4.5,.1),("AA Batteries 8 Pack",5.5,.2)]),
 "Health & Beauty": (2.5, [("Shampoo 400ml",3.8,.45),("Toothpaste 100ml",2,.12),("Shower Gel 250ml",2.2,.28),("Deodorant 150ml",2.8,.17),
                           ("Hand Cream 100ml",3.5,.12),("Pain Relief Tablets 16",1.5,.05),("Sun Cream SPF30 200ml",7.5,.22),("Razors 4 Pack",5.5,.1)]),
 "Clothing": (1.5, [("Cotton T-Shirt",8,.2),("Slim Fit Jeans",25,.7),("Wool Jumper",30,.5),("Waterproof Jacket",45,.9),
                    ("Wellington Boots",22,1.1),("Wool Socks 3 Pack",9,.2),("Winter Scarf",12,.2),("Compact Umbrella",9.5,.35)]),
 "Electronics": (0.8, [("Phone Charger 20W",10,.1),("USB-C Cable 1m",6,.05),("Wireless Earbuds",35,.06),("Power Bank 10000mAh",20,.25),
                       ("Bluetooth Speaker",40,.6),("Electric Kettle 1.7L",25,1.2),("2-Slice Toaster",28,1.5),("Hair Dryer 2000W",22,.7)]),
}

# Define a list of fictional product brands.
BRANDS = ["Ashford Farm","Brookside","Kestrel","Thistle & Co","Harbour Lane","Greenfield","Northgate","Oakley"]

# Generate product records.
# Each catalogue item is assigned two randomly selected brands.
# This creates different product and brand combinations.
prows, n = [], 0
for cat, (pop, items) in CATALOGUE.items():
    for name, price, wt in items:
        for brand in random.sample(BRANDS, 2):
            n += 1

            # Generate a selling price with random variation,
            # rounded to the nearest 5 pence.
            lp = round(price * random.uniform(0.85, 1.3) * 20) / 20              # to the nearest 5p

            # Calculate the unit cost as a percentage of
            # the selling price to ensure a positive profit margin.
            prows.append((f"SKU-{n:04d}", f"{brand} {name}", cat, brand, round(lp * random.uniform(0.55, 0.8), 2), lp, wt))

# Insert inventory records into the inventory table.
# Each record contains store ID, product ID, quantity available
# and reorder level.

# The combination of store_id and product_id forms a
# composite primary key in the inventory table.
con.executemany("INSERT INTO products VALUES (NULL,?,?,?,?,?,?,?)", prows)
PROD = {r[0]: dict(cat=r[1], lp=r[2]) for r in con.execute("SELECT product_id, category, list_price_gbp FROM products")}
PW = {p: CATALOGUE[v["cat"]][0] * random.lognormvariate(0, 0.6) for p, v in PROD.items()}     # long-tailed popularity

# inventory (every store x every product -> compound key)
scale = {"Small": .5, "Medium": .8, "Large": 1.2, "Flagship": 1.8}
con.executemany("INSERT INTO inventory VALUES (?,?,?,?)",
    [(s, p, int(random.randint(0, 250) * scale[STORE[s]["tier"]]), int(random.randint(8, 40) * scale[STORE[s]["tier"]]))
     for s in STORE for p in PROD])
con.commit()

In [ ]:
# orders and order_items
N_ORDERS = 5000
MONTH_W = {1:1.0, 2:0.9, 3:1.0, 4:1.0, 5:0.95, 6:0.95, 7:0.95, 8:0.95, 9:0.95, 10:1.0, 11:1.25, 12:1.5}   # Christmas peak
DOW_W   = [0.9, 0.9, 0.95, 1.0, 1.2, 1.5, 0.8]                                                           # Mon..Sun (Sunday short hours)
TEMP    = [8, 8, 10, 13, 16, 19, 21, 21, 18, 14, 11, 8]                                                  # UK mean daily max by month
TIER_DISC = {"Standard": 0, "Silver": 2, "Gold": 5, "Platinum": 8}
SALES = {11: (15, 25), 12: (5, 15), 1: (10, 25), 7: (10, 20)}   # Black Friday / Christmas / January & summer sales
QTY = {"Food Cupboard": [45,28,15,7,5], "Fresh & Dairy": [45,28,15,7,5], "Drinks": [50,28,12,6,4]}

# Generate every calendar date between the start and end dates.
days = [START + timedelta(i) for i in range((END - START).days + 1)]
dayw = [MONTH_W[d.month] * DOW_W[d.weekday()] for d in days]

# Create an empty list to store generated order timestamps.
stamps = []

# Randomly select order dates using the calculated daily weights.
# Dates with higher weights are more likely to be selected.
for d in random.choices(days, dayw, k=N_ORDERS):
    hrs = list(range(10, 16)) if d.weekday() == 6 else list(range(8, 21))          # Sunday trading hours
    h = random.choices(hrs, [1 + 2 * math.exp(-((x - 17) ** 2) / 8) for x in hrs])[0]   # evening peak
    stamps.append(datetime(d.year, d.month, d.day, h, random.randint(0, 59), random.randint(0, 59)))
stamps.sort()

# Create two empty lists to store the generated records.
# orders stores transaction-level information.
# items stores individual products purchased in each transaction.
orders, items = [], []
for oid, ts in enumerate(stamps, start=1):
    d = ts.date()
    open_ids = [s for s in STORE if STORE[s]["opened"] <= d]
    sid = random.choices(open_ids, [TRAFFIC[STORE[s]["tier"]] for s in open_ids])[0]
    cid = None
    if random.random() < 0.7:                                                       # 70% use a membership card
        for _ in range(8):
            c = random.choice(BY_CITY[STORE[sid]["city"]])
            if CUST[c][1] <= d: cid = c; break
    base = TEMP[d.month - 1] + {"Scotland": -1.5, "Northern Ireland": -1}.get(STORE[sid]["nation"], 0) + (1 if STORE[sid]["city"] == "London" else 0)
    # Calculate transaction temperature using the base temperature,
    # time of day and random variation
    temp = round(base - 3 + 3 * math.cos((ts.hour - 14) / 24 * 2 * math.pi) + random.gauss(0, 2), 1)

    # Randomly assign a payment method using weighted probabilities.
    pay = random.choices(["Contactless Card","Chip & PIN Card","Cash","Mobile Wallet","Gift Card"], [0.45, 0.2, 0.1, 0.22, 0.03])[0]
    rating = random.choices([1,2,3,4,5], [4,6,15,35,40])[0] if random.random() < 0.35 else None

    # Append the order information to the orders list.
    orders.append((oid, cid, sid, ts.isoformat(sep=" "), pay, temp, rating))


    # Randomly determine the number of different product lines
    # purchased in each transaction, between 1 and 8.
    nlines = random.choices(range(1, 9), [22, 24, 18, 13, 9, 6, 4, 2])[0]
    basket = list(dict.fromkeys(random.choices(list(PROD), [PW[p] for p in PROD], k=nlines * 3)))[:nlines]
    for ln, pid in enumerate(basket, start=1):
        qw = QTY.get(PROD[pid]["cat"], [75, 18, 7])
        qty = random.choices(range(1, len(qw) + 1), qw)[0]
        months_left = (END.year - d.year) * 12 + END.month - d.month
        price = round(PROD[pid]["lp"] / (1 + 0.003 * months_left) * 20) / 20           # gentle price inflation over time
        promo = random.uniform(*SALES[d.month]) if d.month in SALES and random.random() < 0.35 else 0
        items.append((oid, ln, pid, qty, price, round(min(40, TIER_DISC[CUST[cid][0]] if cid else 0) + promo, 1)))

# Insert all generated transaction records into the orders table.
# Each order contains customer, store, date, payment,
# temperature and satisfaction information.
con.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?,?)", orders)

# Insert all generated product purchase records into order_items.
# Each record links a product to its corresponding order.
con.executemany("INSERT INTO order_items VALUES (?,?,?,?,?,?)", items)

# Commit the transaction to save all generated records
# permanently to the SQLite database
con.commit()

## Check the database

In [ ]:
import pandas as pd
# PRAGMA integrity_check checks whether the database structure
# is healthy and has no corruption.
# The expected result is "ok".

# PRAGMA foreign_key_check checks whether any foreign-key
# relationships have been broken.
# The expected number of violations is 0.
print("integrity:", con.execute("PRAGMA integrity_check").fetchone()[0],
      "| foreign-key violations:", len(con.execute("PRAGMA foreign_key_check").fetchall()))

# This loop goes through all six tables in the database.
# COUNT(*) returns the number of records (rows) in each table.
# PRAGMA table_info() returns information about the columns.
# This helps us confirm that the tables contain the expected
# number of records and columns.

for t in ["stores","customers","products","orders","order_items","inventory"]:
    print(f"{t:12s}", con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], "rows,",
          len(con.execute(f"PRAGMA table_info({t})").fetchall()), "columns")

print("\nSample: revenue by nation (GBP)")
display(pd.read_sql_query("""
    SELECT s.nation, COUNT(DISTINCT o.order_id) AS orders,
           ROUND(SUM(oi.quantity*oi.unit_price_gbp*(1-oi.discount_pct/100.0)),2) AS revenue_gbp
    FROM orders o JOIN order_items oi USING(order_id) JOIN stores s USING(store_id)
    GROUP BY s.nation ORDER BY revenue_gbp DESC""", con))
print("Sample customers:")
display(pd.read_sql_query("SELECT first_name, last_name, city, postcode_area, email, member_tier FROM customers LIMIT 5", con))

integrity: ok | foreign-key violations: 0
stores       12 rows, 8 columns
customers    2500 rows, 11 columns
products     112 rows, 8 columns
orders       5000 rows, 7 columns
order_items  15157 rows, 6 columns
inventory    1344 rows, 4 columns

Sample: revenue by nation (GBP)


,nation,orders,revenue_gbp
0,England,3571,80535.36
1,Scotland,913,20440.13
2,Wales,350,7229.14
3,Northern Ireland,166,3785.27


Sample customers:


,first_name,last_name,city,postcode_area,email,member_tier
0,Callum,Wilson,Leeds,LS8,callum.wilson8672@example.com,Standard
1,Daniel,Brown,London,W1,daniel.brown6741@example.com,Silver
2,Grace,Smith,Glasgow,G2,grace.smith6515@example.com,Silver
3,Jack,Singh,London,SW9,jack.singh2231@example.com,Platinum
4,Ella,Wilson,Birmingham,B13,ella.wilson5311@example.com,Standard


In [ ]:
# Close the SQLite connection.
# This releases the connection and finishes working with the database.
con.close()
try:
    from google.colab import files
    files.download(DB)
except ImportError:
    print("Saved as", DB)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>